In [ ]:
!pip3 install prophet

In [ ]:
import pandas as pd
# Loading the dataset, organized and structured format
# and perform operations on the right variables

df = pd.read_csv("SUMMER Forecasting - Sheet1.csv")
df = df.rename(columns={"DATE": "ds", "USAGE": "y", "TEMP": "temperature"})
df

In [ ]:
#forecast for 6 months
import numpy as np
from scipy.integrate import trapz
from neuralprophet import NeuralProphet, set_log_level
import pandas as pd

# Data preprocessing so that AI efficiently interprets values and understands what to account for

# Grouping data by date and summing up the values, packages values into one 
df = df.groupby('ds').sum().reset_index()

# Disable logging messages unless there is an error
set_log_level("ERROR")

# Creating the NeuralProphet model
m = NeuralProphet(
    
    # Disable trend changepoints to account for outliers 
    n_changepoints=10,

    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    
    n_forecasts=183 #forecast for 180 days in the future with 180 forecast lines
)
m.set_plotting_backend("plotly")

# Adding temperature as a lagged regressor, the model is able to percieve that temperature will become another factor 
m.add_lagged_regressor("temperature")

# Training the model: supervised learning, training on historical data to draw patterns
metrics = m.fit(df)

# Adding temperatures to the future data frame of future dates 
future_temps = [57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45,57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45,57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45,57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45,57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45,57,55,53,54,50,53,52,52,46,46,51,53,46,48,51,46,43,46,46,51,56,52,49,46,47,45,47,57,45, 45, 45, 45]  # Replace with your list of temperatures
df_future = m.make_future_dataframe(df, n_historic_predictions=True, periods=183)
df_future['temperature'] = df['temperature'].append(pd.Series(future_temps)).reset_index(drop=True)  # Add the new 'temperature' column to the dataframe

# Predict the future,  deep learning architechture uses statistical calculations 
forecast = m.predict(df_future)

#Plot, shows clear visualizations to users

fig1 = m.plot(forecast)
fig1.show()

# Calculating the net energy consumption
integral = forecast['yhat1'].sum()


print("Net energy consumption for the 12 months is", integral)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt



df['ds'] = pd.to_datetime(df['ds'])

# Sort the DataFrame by the 'ds' column (date) for a chronological plot
df = df.sort_values(by='ds')


plt.figure(figsize=(12, 6))  # Adjust the figure size as needed
plt.plot(df['ds'], df['y'], label='Usage')
plt.plot(df['ds'], df['temperature'], label='Temperature')
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Usage and Temperature Over Time')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()